# 5.13 資料或模型變大，哪個值得花預算？


小模型看少資料和大模型看多資料相比，後者如果更好，我們仍不知道功勞來自更多參數、更多題目還是更久訓練。要回答「預算先花在哪」，先把模型容量與資料量分開變化，像試食譜時分別改變食材與烹調時間，而不是把所有條件一次換掉。

最小設計是兩種寬度乘兩種資料量，共四格。[4.8](https://birdhackor.github.io/tiny-perceptron-vlm/4.8.html) 說明寬度會改參數成本；這裡width8與16、不同記錄16與64形成組合。每格計劃跑100步，卻還要明示每步多少筆與有效答案，才能知道究竟看了多少token。token是模型處理的一個文字片段，目前可先當作字，後面第6章會改變拆法。這裡每個「有效答案」指一個有正確目標、實際參與訓練誤差的下一token預測位置，不是一道完整問題。例如前文「貓」要接「看」，再以「貓看」預測「狗」，就是兩個有效位置；用來湊齊長度的空格若設為忽略，就不計入。本節的有效token預算就是這些有效答案位置的總數，可回看[5.7的逐位置答案](https://birdhackor.github.io/tiny-perceptron-vlm/5.7.html)。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig

for width in [8, 16]:
    parameters = TinyLM(ModelConfig(width=width)).description()["parameters"]
    for records in [16, 64]:
        batch_size, answers_per_record, steps = 4, 8, 100
        print(
            {
                "width": width,
                "不同記錄": records,
                "參數格數": parameters,
                "更新步數": steps,
                "有效答案預算": batch_size * answers_per_record * steps,
                "品質": "尚待訓練與量測",
            }
        )

輸入是實驗計劃，不是現成的訓練資料。`records` 指不同例子數，batch_size是每步四筆，假定每筆八個有效答案，所以每格共3200答案；16筆資料會被重複更多次，64筆則包含更多不同題。程序只建立模型數參數並印四份計畫，品質欄保留未測狀態，不能從參數數字推定分數。

「公平預算」有多種定義。固定步數方便看每次更新效果；固定有效token數讓見到的監督總量相近；固定時間或FLOPs則更直接限制計算成本。FLOPs是浮點運算次數，例如一次乘法與一次加法各算一個。D是輸入特徵數，H是輸出特徵數；密集Linear用一張H橫排、D欄的權重表，把D個輸入數加權混合成H個輸出數，每個輸出都使用全部輸入。處理一個位置約需2×D×H次運算，這裡忽略每個輸出混合完再加上一個可調數字的成本（這個數字叫偏置），以及其他較小成本：8進8出約128次，16進16出約512次。模型變寬會同時放大兩邊，所以相同token預算也不等於相同計算預算。耗時測量可回看 [5.9](https://birdhackor.github.io/tiny-perceptron-vlm/5.9.html)。

還要用共同的驗證與測試資料評品質，才能把四格的變化歸因於訓練條件。若大資料格包含驗證題，或大模型多選了幾次最好種子，比較會混入別的差異，見 [5.10](https://birdhackor.github.io/tiny-perceptron-vlm/5.10.html)、[5.15](https://birdhackor.github.io/tiny-perceptron-vlm/5.15.html)。保存數據版本與實際有效token數，比只寫「100步」更能讓別人複覈。

較大模型與更多資料如何共同影響品質，常被研究成scaling law（規模與表現的經驗規律）。我們的四個極小計劃點還不能推論大型規律，何況尚未量到品質；它們的用途是練習控制變量與明確預算，讓下一次實驗能回答一個具體問題。

練習保持每步四筆，假設某格每筆有效答案只有四個，想維持3200token預算應跑幾步？先算 `3200/(4×4)=200`，再改該格記錄核對。若仍寫100步，那格只有1600答案，已經不是同樣監督預算。將計劃中的假設換成實測長度統計，纔會成為真實資源記錄。
